# Kadane (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/Kadane_cpp.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `Kadane`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/Kadane_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/Kadane_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `RecursiveSum` (PascalCase).

## Kadane

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Una sola scansione: $cur$ è la somma massima di un sottoarray che **termina** nella posizione corrente e $cl$ il suo primo indice; se $cur<0$ si riparte da $a[i]$, altrimenti si estende. $\Theta(n)$ passi, memoria extra $\Theta(1)$. (Nel capitolo 18 si trova la versione compatta che restituisce solo la somma.)

Sull'esempio della compravendita di azioni (variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) si stampa la tabella del capitolo: una riga per iterazione, con i valori alla fine dell'iterazione (indici da 0, la riga $i=0$ è l'inizializzazione). Il risultato $(7,10,43)$, cioè $(8,11,43)$ con indici da 1, coincide con quello dell'algoritmo divide et impera.

In [1]:
%%writefile kadane.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

struct Triple
{
    int i, j, sum;
};

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Restituisce (bl, br, best): un sottoarray di somma massima di a[0..n-1]
Triple Kadane(int *a, int n)
{
    int best = a[0];
    int bl = 0;
    int br = 0;
    int cur = a[0];             // somma massima di un sottoarray che termina in i
    int cl = 0;
    int i = 1;
    while (i < n)
    {
        if (cur < 0)
        {
            cur = a[i];         // riparte
            cl = i;
        }
        else
        {
            cur = cur + a[i];   // estende
        }
        if (cur > best)
        {
            best = cur;
            bl = cl;
            br = i;
        }
        i = i + 1;
    }
    return {bl, br, best};
}

// Come sopra, ma stampa una riga per iterazione
Triple KadaneTrace(int *a, int n)
{
    int best = a[0], bl = 0, br = 0;
    int cur = a[0], cl = 0;
    printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)\n", 0, a[0], "inizializzazione", cur, cl, best, bl, br);
    for (int i = 1; i < n; i++)
    {
        const char *action;
        if (cur < 0)
        {
            cur = a[i];
            cl = i;
            action = "riparte (cur<0)";
        }
        else
        {
            cur = cur + a[i];
            action = "estende";
        }
        if (cur > best)
        {
            best = cur;
            bl = cl;
            br = i;
        }
        printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)\n", i, a[i], action, cur, cl, best, bl, br);
    }
    return {bl, br, best};
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    int n = 16;
    printf("a = %s\n", Fmt(a, n).c_str());
    printf("%3s %5s  %-24s %4s %3s %5s  (bl, br)\n", "i", "a[i]", "azione", "cur", "cl", "best");
    Triple res = KadaneTrace(a, n);
    printf("risultato (%d, %d, %d)   (%d, %d, %d con indici da 1)\n", res.i, res.j, res.sum, res.i + 1, res.j + 1, res.sum);
    int b[] = {-2, 5, -1, 3};
    Triple t = Kadane(b, 4);
    printf("a = %s  Kadane(a, 4) = (%d, %d, %d)\n", Fmt(b, 4).c_str(), t.i, t.j, t.sum);
    return 0;
}
#endif

Writing kadane.cpp


In [2]:
!g++ -std=c++17 -O2 -o kadane kadane.cpp && ./kadane

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  azione                    cur  cl  best  (bl, br)
  0    13  inizializzazione           13   0    13  (0, 0)
  1    -3  estende                    10   0    13  (0, 0)
  2   -25  estende                   -15   0    13  (0, 0)
  3    20  riparte (cur<0)            20   3    20  (3, 3)
  4    -3  estende                    17   3    20  (3, 3)
  5   -16  estende                     1   3    20  (3, 3)
  6   -23  estende                   -22   3    20  (3, 3)
  7    18  riparte (cur<0)            18   7    20  (3, 3)
  8    20  estende                    38   7    38  (7, 8)
  9    -7  estende                    31   7    38  (7, 8)
 10    12  estende                    43   7    43  (7, 10)
 11    -5  estende                    38   7    43  (7, 10)
 12   -22  estende                    16   7    43  (7, 10)
 13    15  estende                    31   7    43  (7, 10)
 14    -4  estende                   

**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): Kadane riparte a ogni passo e trova il massimo elemento.

In [3]:
%%writefile prova_kadane.cpp
#define NOMAIN
#include "kadane.cpp"

int main()
{
    // Prova con un array di soli numeri negativi: Kadane riparte a ogni passo e trova il massimo elemento.
    int c[] = {-3, -1, -2, -5};
    Triple t = Kadane(c, 4);
    printf("(%d, %d, %d)\n", t.i, t.j, t.sum);
    return 0;
}

Writing prova_kadane.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_kadane prova_kadane.cpp && ./prova_kadane

(1, 1, -1)


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).